# Q学習

Q 学習は、各状態で各行動がどれくらい将来の得につながるかを直接学びながら、最終的に最適行動を目指す方法です。状態価値を経由せず、その場で行動どうしを比べられるので、モデルなし制御の代表例になります。


## 参考動画（外部）

授業本編ではなく、別の説明で見直したいときの参考材料です。

- [Q学習の解説](https://www.youtube.com/watch?v=JPt5JYmcngc)


## `max` を入れた瞬間に「最善ならどうなるか」を学び始める

Q 学習の特徴は、次状態で実際に取った行動ではなく、そこで取りうる最善行動の価値を目標値に入れることです。つまり、実際には探索で少し遠回りしていても、学習の中では「次は最善を選べたとしたら」と仮定して値を更新します。


ここで見たいのは、`Q(s,a)` の更新式そのものより、なぜ `max` が policy の外側にいるのかです。実際の行動列と、学習が目指す理想の行動列が一致しなくてよいのが Q 学習の強さであり、同時に強気すぎる見積もりの入口にもなります。


## 読み進める視点

参照用の return、1 回の Q 更新、いま一番よさそうな行動の選び方、探索と活用の切り替え、最後に得られた方策の読み方を順に見ます。


この notebook の主題は、表形式 Q 学習の骨格です。あとで deep RL を読むときも、target の中身は結局ここで見た `r + \gamma \max Q` から始まっています。


## 読み方の軸

Q 学習では「実際に何をしたか」より「そこから最善ならどうするか」を更新します。この差が、SARSA との対比でいちばん重要です。


## 検証 1: Q学習の更新準備

Q学習の更新式へ入る前に、まず長期の報酬を1本の数値として見る感覚をそろえます。


In [ ]:
rewards = [0.0, 1.0, 0.2, 1.4]
gamma = 0.91
g = 0.0
for r in reversed(rewards):
    g = r + gamma * g
print('task = q-learning', 'reference_return=', round(g, 6))

次のセルで、この長期感覚を `max` を含む更新規則へ接続します。



## 検証 2: ベルマン更新を1回行う

次に、価値更新を1ステップだけ計算します。1回更新でも、「次に一番よさそうな行動を見て今を決める」構造は十分に見えてきます。


In [ ]:
v_next = {'s0': 0.4, 's1': 0.8}
reward = {'left': 0.2, 'right': 1.0}
trans = {'left': 's0', 'right': 's1'}
v_s = max(reward[a] + gamma * v_next[trans[a]] for a in ['left', 'right'])
print('updated V(s)=', round(v_s, 6))

ベルマン更新は『今の価値』を『次状態でどれだけ得しそうか』で再定義する操作です。この再帰が強化学習の中心です。



## 定義の確認

1. $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma\max_{a'}Q(s',a') - Q(s,a)]$


## 検証 3: Q値更新を比較する

ここで Q学習の更新式をコードに写し、数値の動きを確認します。式を読むだけでは掴みにくい、「理想の次の一手を仮定して今を直す」感覚を得る段階です。


In [ ]:
Q = {('s0','left'): 0.3, ('s0','right'): 0.1, ('s1','left'): 0.5, ('s1','right'): 0.7}
alpha = 0.2
r, s, a, s_next = 1.0, 's0', 'right', 's1'
td_target = r + gamma * max(Q[(s_next,'left')], Q[(s_next,'right')])
Q[(s,a)] += alpha * (td_target - Q[(s,a)])
print('Q(s0,right)=', round(Q[(s,a)], 6))

更新後の値が過去の値とどれだけ違うかは、学習率と TD 誤差で決まります。ここが「どれくらい強く理想側へ寄せるか」の調整ポイントです。



## 検証 4: 探索と活用の切り替え

次に、探索率を変えたときの行動選択を見ます。Q 学習では、実際の行動は探索で揺れてよい一方、更新では理想の次の一手を仮定するので、この切り分けが大事です。


In [ ]:
import random

random.seed(7)

def choose_action(q_left, q_right, epsilon):
    if random.random() < epsilon:
        return random.choice(['left', 'right'])
    return 'left' if q_left >= q_right else 'right'

samples_high_eps = [choose_action(0.4, 0.7, 0.5) for _ in range(8)]
samples_low_eps = [choose_action(0.4, 0.7, 0.1) for _ in range(8)]
print('epsilon=0.5 ->', samples_high_eps)
print('epsilon=0.1 ->', samples_low_eps)


探索率は固定せず、学習段階に応じて減衰させるのが一般的です。初期は広く試し、後半でよさそうな行動の活用へ寄せます。



## 検証 5: 方策評価の簡易チェック

最後に、方策の平均報酬を簡易的に比較します。更新式が分かっても、最終的にどんな行動方針ができたかを見ないと学習は評価できません。


In [ ]:
episode_rewards = [1.2, 0.8, 1.5, 1.1, 1.4]
avg_reward = sum(episode_rewards) / len(episode_rewards)
variance = sum((r - avg_reward) ** 2 for r in episode_rewards) / len(episode_rewards)
print('avg =', round(avg_reward, 4))
print('var =', round(variance, 4))

平均だけでなく分散を見ると、方策の安定性も評価できます。たまに当たるだけなのか、安定して強いのかを分けて見られるからです。



## この節の要点

1. Q 学習は `Q(s,a)` を直接学ぶ。
2. 目標値に `max_{a'} Q(s',a')` を入れるので、実際の探索行動と学習目標を切り分けられる。
3. `max` の強気さが、後で SARSA との振る舞いの差を生む。
